03 - Data Modeling

Purpose: This notebook should document and validate the PostgreSQL warehouse.

This file will help in:

- table counts
- null checks
- foreign key validation checks
- date range
- sample joined fact + dimensions


# Importing Libraries


In [9]:
from sqlalchemy import text, inspect
import pandas as pd

# Setup


In [10]:
import sys

sys.path.append("..")
from src.load_postgres import get_engine

# Connect to PostgreSQL


In [11]:
engine = get_engine()

In [12]:
try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT 1"))
        print("Database Connection successful")
        print(f"Result of the text query: {result.scalar()} ")
except Exception as e:
    raise ValueError(f"Error: {e}")

Database Connection successful
Result of the text query: 1 


# Warehouse Table Counts


In [13]:
inspector = inspect(engine)
table_names = inspector.get_table_names()
table_names

['dim_artists',
 'dim_tracks',
 'dim_users',
 'fact_listening_events',
 'dim_dates']

In [ ]:
warehouse_dict = []
with engine.connect() as connection:
    for table in table_names:
        query = text(f"SELECT COUNT(*) FROM {table}")
        try:
            row_count = connection.execute(query).scalar()
        except Exception as e:
            raise ValueError(f"Error:{e}")

        warehouse_dict.append({"table_name": table, "row_count": row_count})

warehouse = pd.DataFrame(warehouse_dict)
warehouse

,table_name,row_count
0,dim_artists,176697
1,dim_tracks,1503135
2,dim_users,992
3,fact_listening_events,19098642
4,dim_dates,1589


# Star Schema Validation


In [37]:
queries = {
    "fact_nulls": """
        SELECT COUNT(*) FROM fact_listening_events 
        WHERE user_id IS NULL OR artist_key IS NULL OR track_key IS NULL 
           OR listened_at IS NULL OR listened_date IS NULL
    """,
    "missing_users": """
        SELECT COUNT(*) FROM fact_listening_events f 
        LEFT JOIN dim_users u ON f.user_id = u.user_id 
        WHERE u.user_id IS NULL
    """,
    "missing_artists": """
        SELECT COUNT(*) FROM fact_listening_events f 
        LEFT JOIN dim_artists a ON f.artist_key = a.artist_key 
        WHERE a.artist_key IS NULL
    """,
    "missing_tracks": """
        SELECT COUNT(*) FROM fact_listening_events f 
        LEFT JOIN dim_tracks t ON f.track_key = t.track_key 
        WHERE t.track_key IS NULL
    """,
    "missing_dates": """
        SELECT COUNT(*) FROM fact_listening_events f 
        LEFT JOIN dim_dates d ON f.listened_date = d.date_key 
        WHERE d.date_key IS NULL
    """,
}

results_list = []

In [38]:
with engine.connect() as connection:
    for check_name, sql_string in queries.items():
        try:
            row_count = connection.execute(text(sql_string)).scalar()
            results_list.append({"check_name": check_name, "row_count": row_count})
        except Exception as e:
            raise ValueError(f"Error:{e}")


results = pd.DataFrame(results_list)
results

,check_name,row_count
0,fact_nulls,0
1,missing_users,0
2,missing_artists,0
3,missing_tracks,0
4,missing_dates,0


# Date Range Validation


In [25]:
query = text(
    "SELECT MIN(listened_at) AS min_listened_at, MAX(listened_at) AS max_listened_at FROM  fact_listening_events;"
)

In [ ]:
with engine.connect() as connection:
    try:
        value = connection.execute(query).fetchone()
        print(f"Date Range: {value[0]} to {value[1]}")
    except Exception as e:
        raise ValueError(f"Error:{e}")

Date Range: 2005-02-13 19:00:07-05:00 to 2013-09-29 14:32:04-04:00


# Sample Joined Records


In [39]:
query = text("""
    SELECT 
        f.event_id,
        f.user_id,
        u.country,
        u.age,
        f.listened_at,
        a.artist_name,
        t.track_name,
        d.year,
        d.month_name,
        d.day_name
    FROM fact_listening_events f 
    LEFT JOIN dim_users u    ON f.user_id = u.user_id 
    LEFT JOIN dim_artists a  ON f.artist_key = a.artist_key 
    LEFT JOIN dim_tracks t   ON f.track_key = t.track_key 
    LEFT JOIN dim_dates d    ON f.listened_date = d.date_key 
    LIMIT 5;
    """)

In [40]:
with engine.connect() as connection:
    try:
        df = pd.read_sql_query(query, con=connection)
        display(df)
    except Exception as e:
        raise ValueError(f"Error:{e}")

,event_id,user_id,country,age,listened_at,artist_name,track_name,year,month_name,day_name
0,4154954,user_000210,United Kingdom,42.0,2007-06-05 12:11:05-04:00,July Skies,Farmers And Villagers Living Within The Shadow...,2007,June,Tuesday
1,11153646,user_000595,Russian Federation,NaN,2007-10-17 10:01:50-04:00,July Skies,Farmers And Villagers Living Within The Shadow...,2007,October,Wednesday
2,15444038,user_000800,Russian Federation,NaN,2006-12-01 17:43:26-05:00,July Skies,Farmers And Villagers Living Within The Shadow...,2006,December,Friday
3,15444869,user_000800,Russian Federation,NaN,2006-11-22 17:28:03-05:00,July Skies,Farmers And Villagers Living Within The Shadow...,2006,November,Wednesday
4,945006,user_000038,Russian Federation,29.0,2009-03-31 08:56:48-04:00,In Extremo,Como Poden,2009,March,Tuesday


# Modeling Decisions

- The warehouse uses a star schema because the project is focused on analytics queries, KPIs, retention analysis, funnel analysis, segmentation, and replay modeling.
- `fact_listening_events` stores one row per cleaned listening event.
- `dim_users` stores user profile attributes.
- `dim_artists` stores one row per analysis-ready artist key.
- `dim_tracks` stores one row per analysis-ready track key.
- `dim_dates` supports time-based grouping by day, week, month, quarter, and year.
- Missing raw `artist_id` and `track_id` values were retained using fallback keys.
- Long name-based fallback keys were compacted with deterministic hashing before warehouse loading to keep database indexes stable.
- Indexes were separated from the base schema and created after loading to improve full-load performance.


# Phase 3 Summary

- Phase 3 successfully loaded the cleaned Last.fm listening data into a PostgreSQL analytics warehouse.
- The warehouse contains one fact table with 19,098,642 validated listening events and four supporting dimension tables for users, artists, tracks, and dates.
- All referential integrity checks passed. Every fact row maps to a valid user, artist, track, and calendar date.
- PostgreSQL stores `listened_at` as `TIMESTAMPTZ`, so displayed values may appear in the local timezone. These correspond to the cleaned UTC range from 2005-02-14 to 2013-09-29.
- Final warehouse counts:
  - `dim_users`: 992 rows
  - `dim_artists`: 176,697 rows
  - `dim_tracks`: 1,503,135 rows
  - `dim_dates`: 1,589 rows
  - `fact_listening_events`: 19,098,642 rows
- Validation checks confirmed that required fact fields are complete and that all fact rows have matching records in the user, artist, track, and date dimensions.
